# Top exit velo + top 5 fastest pitches

Two TrackMan CSVs in, two tables out.

In [2]:
import polars as pl
from pathlib import Path

pl.Config.set_tbl_cols(20)  # show every column in the tables

# the two TrackMan CSVs (swap in your own)
CSV_1 = Path("/Users/matt/Desktop/delispice/backend/notebooks/LiveBpPitching_2026-10-17T003138_verified.csv")
CSV_2 = Path("/Users/matt/Desktop/delispice/backend/notebooks/LiveBpPitching_2026-10-17T185526_verified.csv")

# force the numeric columns to floats so a mostly-empty column can't come in as text
NUM = ["RelSpeed", "InducedVertBreak", "HorzBreak", "SpinRate", "SpinAxis",
       "ExitSpeed", "Angle", "Distance"]

df = pl.concat(
    [pl.read_csv(p, infer_schema_length=None, schema_overrides={c: pl.Float64 for c in NUM})
     .rename({"HitType": "TaggedHitType"}, strict=False)  # Live BP files call it HitType
     for p in (CSV_1, CSV_2)],
    how="diagonal_relaxed",  # tolerate column order differences between files
)

# same hit types as the parquets: no contact (null here) -> "Undefined"
HIT_TYPES = ["GroundBall", "LineDrive", "FlyBall", "Popup", "Bunt"]
df = df.with_columns(
    pl.when(pl.col("TaggedHitType").is_in(HIT_TYPES))
    .then(pl.col("TaggedHitType"))
    .otherwise(pl.lit("Undefined"))
    .alias("TaggedHitType")
)

print(f"{df.height} pitches from {CSV_1.name} + {CSV_2.name}")
df["TaggedHitType"].value_counts(sort=True)

766 pitches from LiveBpPitching_2026-10-17T003138_verified.csv + LiveBpPitching_2026-10-17T185526_verified.csv


TaggedHitType,count
str,u32
"""Undefined""",689
"""GroundBall""",37
"""FlyBall""",20
"""LineDrive""",14
"""Popup""",6


In [5]:
# hardest-hit ball across every hit type (no PitchCall in these files, so hit type marks the batted balls)
top_ev = (
    df.filter(pl.col("TaggedHitType") != "Undefined", pl.col("ExitSpeed").is_not_null())
    .sort("ExitSpeed", descending=True)
    .head(10)
    .select(
        "Batter",
        "Pitcher",
        pl.col("TaggedPitchType").alias("Pitch"),
        pl.col("RelSpeed").round(1).alias("Pitch Velo"),
        pl.col("ExitSpeed").round(1).alias("EV"),
        pl.col("Angle").round(1).alias("LA"),
        pl.col("Distance").round(0).alias("Distance"),
        pl.col("TaggedHitType").alias("Hit Type"),
    )
)
top_ev

Batter,Pitcher,Pitch,Pitch Velo,EV,LA,Distance,Hit Type
str,str,str,f64,f64,f64,f64,str
"""Orr, Robert""","""Fausset, Cameron""","""Fastball""",91.9,105.3,7.6,161.0,"""GroundBall"""
"""Covarrubias, Abbrie""","""Schroeder, Jackson""","""Sinker""",92.8,104.2,6.5,179.0,"""GroundBall"""
"""Covarrubias, Abbrie""","""Fausset, Cameron""","""Sinker""",91.9,103.9,14.3,341.0,"""LineDrive"""
"""Lamb, Andrew""","""Gubitosi, Anthony""","""Slider""",81.3,103.5,-4.2,22.0,"""GroundBall"""
"""Souza, Jace""","""Gubitosi, Anthony""","""Slider""",80.6,103.0,4.3,149.0,"""GroundBall"""
"""Ohira, Jett""","""Gubitosi, Anthony""","""Fastball""",88.9,102.8,18.6,379.0,"""LineDrive"""
"""Lamb, Andrew""","""Fausset, Cameron""","""Fastball""",91.9,102.8,8.4,166.0,"""GroundBall"""
"""Lamb, Andrew""","""Matson, Sax""","""Slider""",79.7,102.6,8.1,151.0,"""GroundBall"""
"""Takeuchi, Kevin""","""White, Bo""","""Sinker""",90.9,101.9,5.9,148.0,"""GroundBall"""


In [4]:
# 5 fastest pitches with their movement (IVB / HB in inches) and spin
top_velo = (
    df.filter(pl.col("RelSpeed").is_not_null())
    .sort("RelSpeed", descending=True)
    .head(5)
    .select(
        "Pitcher",
        pl.col("TaggedPitchType").alias("Pitch"),
        pl.col("RelSpeed").round(1).alias("Velo"),
        pl.col("InducedVertBreak").round(1).alias("IVB"),
        pl.col("HorzBreak").round(1).alias("HB"),
        pl.col("SpinRate").round(0).alias("Spin"),
        pl.col("SpinAxis").round(0).alias("Axis"),
    )
)
top_velo

Pitcher,Pitch,Velo,IVB,HB,Spin,Axis
str,str,f64,f64,f64,f64,f64
"""Michel, Matias""","""Fastball""",96.2,15.7,15.2,2341.0,224.0
"""Michel, Matias""","""Fastball""",95.7,19.1,16.4,2269.0,220.0
"""Michel, Matias""","""Fastball""",95.3,16.2,17.3,2254.0,227.0
"""Michel, Matias""","""Fastball""",95.2,12.1,15.6,2200.0,232.0
"""Michel, Matias""","""Fastball""",95.0,12.8,15.7,2203.0,231.0
